# Compile raw responses into CSVs

- Turns each run's `<run_id>.raw.jsonl` (written by `run_scrapers.ipynb`) into a folder of CSVs, one file per county: `data/csv/<run_id>/<county_fips>.csv`
- One row per plan per scenario
- Does not require network access
- The row format is defined in this notebook, under **Row format**
- Edit the functions there to add or change columns, check the preview, then compile with `OVERWRITE = True`
- Dask parses the raw file in parallel blocks, so the multi-gigabyte JSON never has to fit in memory; only the resulting rows do
- Everything compiled loads as a single dask dataframe, one partition per county:

```python
dd.read_csv("data/csv/*/*.csv", dtype={"zip_code": str, "county_fips": str, "plan_id": str}, assume_missing=True)
```

## Configuration

- `run_ids` lists the raw files to compile, each with its relative path
- The commented line below it builds the list from every `.jsonl` file in a folder instead
- Each raw file gets its own folder under `CSV_DIR`, named after the file

In [ ]:
from pathlib import Path

# run_ids = ["data/results/20261118T223104Z_AR.raw.jsonl"]  # the raw files to compile, each with its path
run_ids = sorted(str(p) for p in Path("data/results").glob("*.jsonl"))  # or: every .jsonl file in a folder

CSV_DIR = Path("data/csv")  # one sub-folder per raw file, one CSV per county in it
OVERWRITE = False           # False = skip raw files that already have a CSV folder
BLOCKSIZE = "64MB"          # raw text parsed per parallel task

## Row format

- Each raw line holds one response plus the scenario and request that produced it: `{run_id, retrieved_at_utc, source, url, scenario, request, response}`
- `iter_plans(entry)` (from `scrapers`) yields each plan object in the response, using the parser of the site it came from (`scrapers/<site>/parser.py`)
- `scenario_columns` gives the columns every site shares: provenance, location, household
- One `<site>_rows` function per scraper module turns an entry into rows, one per plan, by adding the plan columns
- All `<site>_rows` functions must return the same keys
- Runs quote each county once, so there is one set of rows per county
- `county_fips` is the place quoted; `zip_code` is just the ZIP sent with the request (any one ZIP of that county)
- Money is monthly USD unless the column says otherwise
- Deductible and out-of-pocket maximum are the in-network, per-person amounts

In [16]:
from scrapers import iter_plans, iter_raw


def to_number(value):
    """Parse a money-ish value ("1,750", "$799.55", 799.55, "", None) into a float."""
    if value is None or isinstance(value, bool):
        return None
    if isinstance(value, (int, float)):
        return float(value)
    try:
        return float(str(value).replace("$", "").replace(",", "").strip())
    except ValueError:
        return None


def scenario_columns(entry: dict) -> dict:
    """Who asked, where and when: the same for every plan in the response."""
    scenario = entry["scenario"]
    location, members = scenario["location"], scenario["members"]
    applicant = members[0]
    return {
        "run_id": entry["run_id"],
        "retrieved_at_utc": entry["retrieved_at_utc"],
        "source": entry["source"],
        "source_url": entry["url"],
        "state": location["state"],
        "zip_code": location["zip_code"],
        "county_fips": location["county_fips"],
        "coverage_year": scenario["coverage_year"],
        "household_size": len(members),
        "household_income": scenario["household_income"],
        "applicant_age": applicant["age"],
        "applicant_dob": applicant["dob"],
        "applicant_gender": applicant.get("gender"),
        "has_spouse": any(m["relationship"] == "SPOUSE" for m in members),
        "num_dependents": sum(m["relationship"] == "CHILD" for m in members),
    }

### HealthCare.gov (`federal`)

In [17]:
def federal_cost(entries, kind=None):
    """Per-person in-network amount from a plan's `deductibles` or `moops` list.

    `kind` picks the deductible type by its label ("combined", "medical", "drug"); None takes any.
    Prefers the entry flagged `individual`, then "Family Per Person".
    """
    entries = [e for e in entries or [] if e.get("network_tier", "In-Network") == "In-Network"]
    if kind == "combined":
        entries = [e for e in entries if "combined" in e.get("type", "").lower()]
    elif kind:
        entries = [e for e in entries if kind in e.get("type", "").lower() and "combined" not in e.get("type", "").lower()]
    entries.sort(key=lambda e: (e.get("individual") is not True, e.get("family_cost") != "Family Per Person"))
    return to_number(entries[0].get("amount")) if entries else None


def federal_rows(entry: dict) -> list[dict]:
    who = scenario_columns(entry)
    rows = []
    for plan in iter_plans(entry):
        premium = to_number(plan.get("premium"))
        after_credit = to_number(plan.get("premium_w_credit"))
        combined = federal_cost(plan.get("deductibles"), "combined")
        rows.append({
            **who,
            "plan_id": plan.get("id", "").strip(),
            "plan_name": plan.get("name", "").strip(),
            "issuer": (plan.get("issuer") or {}).get("name", "").strip(),
            "metal_level": plan.get("metal_level", "").strip().upper(),
            "plan_type": plan.get("type", "").strip(),
            "premium_monthly": premium,
            "premium_after_credit_monthly": after_credit,
            "aptc_monthly": None if premium is None or after_credit is None else round(premium - after_credit, 2),
            "deductible_individual": combined if combined is not None else federal_cost(plan.get("deductibles"), "medical"),
            "drug_deductible_individual": None if combined is not None else federal_cost(plan.get("deductibles"), "drug"),
            "deductible_combined_med_drug": combined is not None,
            "moop_individual": federal_cost(plan.get("moops")),
            "hsa_eligible": plan.get("hsa_eligible"),
            "currency": "USD",
        })
    return rows

### Virginia (`virginia`) and Maryland (`maryland`)

In [18]:
def virginia_rows(entry: dict) -> list[dict]:
    who = scenario_columns(entry)
    rows = []
    for plan in iter_plans(entry):  # only the getIndividualPlans response of each scenario has plans
        costs = plan.get("planCosts") or {}
        cost = lambda key: to_number((costs.get(key) or {}).get("inNetworkInd"))
        combined = cost("DEDUCTIBLE_INTG_MED_DRUG")
        hsa = plan.get("hsa")
        rows.append({
            **who,
            "plan_id": str(plan.get("issuerPlanNumber", "")).strip(),
            "plan_name": str(plan.get("name", "")).strip(),
            "issuer": str(plan.get("issuer", "")).strip(),
            "metal_level": str(plan.get("level", "")).strip().upper(),
            "plan_type": str(plan.get("networkType", "")).strip(),
            "premium_monthly": to_number(plan.get("premiumBeforeCredit")),
            "premium_after_credit_monthly": to_number(plan.get("premiumAfterCredit")),
            "aptc_monthly": to_number(plan.get("aptc")),
            "deductible_individual": combined if combined is not None else cost("DEDUCTIBLE_MEDICAL"),
            "drug_deductible_individual": None if combined is not None else cost("DEDUCTIBLE_DRUG"),
            "deductible_combined_med_drug": combined is not None,
            "moop_individual": cost("MAX_OOP_INTG_MED_DRUG"),
            "hsa_eligible": None if hsa is None else str(hsa).strip().lower() == "yes",
            "currency": "USD",
        })
    return rows


def maryland_rows(entry: dict) -> list[dict]:
    # Untested on live data: the site has refused this client (HTTP 403) since 2026-09-30.
    per_person = lambda text: to_number(str(text or "").split("per person")[0])  # "$10600 per person | $21200 per group"
    who = scenario_columns(entry)
    rows = []
    for plan in iter_plans(entry):
        premium = to_number(plan.get("basePremium"))
        after_credit = to_number(plan.get("premiumDouble"))  # after APTC and Maryland's state subsidy
        hsa = str(plan.get("hsaQualifiedText", "")).strip().lower()
        rows.append({
            **who,
            "plan_id": str(plan.get("planDetailsDisplayId", "")).split("-", 1)[-1].replace("-", ""),  # '2026-72375MD0070031-01'
            "plan_name": str(plan.get("planName", "")).strip(),
            "issuer": str(plan.get("carrierName", "")).strip(),
            "metal_level": str(plan.get("metalLevel", "")).strip().upper(),
            "plan_type": str(plan.get("planType", "")).strip(),
            "premium_monthly": premium,
            "premium_after_credit_monthly": after_credit,
            "aptc_monthly": None if premium is None or after_credit is None else round(premium - after_credit, 2),
            "deductible_individual": per_person(plan.get("annualDeductible")),
            "drug_deductible_individual": None,  # the site reports a single annual deductible
            "deductible_combined_med_drug": None,
            "moop_individual": per_person(plan.get("annualoutOfPocketMax")),
            "hsa_eligible": hsa.endswith("yes") if hsa else None,
            "currency": "USD",
        })
    return rows

In [19]:
import json

ROWS = {"federal": federal_rows, "virginia": virginia_rows, "maryland": maryland_rows}  # scraper module -> its row function


def entry_to_rows(entry: dict) -> list[dict]:
    return ROWS[entry["source"]](entry)

## Preview

- Shows the rows from the first response with plans in the first raw file
- Check the columns here before compiling everything

In [20]:
import pandas as pd


def sample_rows(raw_file: str) -> list[dict]:
    """Rows from the first response in the raw file that has any plans."""
    return next((rows for rows in map(entry_to_rows, iter_raw(raw_file)) if rows), [])


preview = pd.DataFrame(sample_rows(run_ids[0])) if run_ids else pd.DataFrame()
print(f"{len(run_ids)} raw files to compile; {len(preview.columns)} columns")
preview.head()

1 raw files to compile; 29 columns


,run_id,retrieved_at_utc,source,source_url,state,zip_code,county_fips,coverage_year,household_size,household_income,...,plan_type,premium_monthly,premium_after_credit_monthly,aptc_monthly,deductible_individual,drug_deductible_individual,deductible_combined_med_drug,moop_individual,hsa_eligible,currency
0,20261118T223104Z_AR,2026-11-18T22:31:04Z,federal,https://marketplace-int.api.healthcare.gov/api...,AR,71635,05003,2026,1,159600,...,PPO,254.18,254.18,0.0,10600.0,None,True,10600.0,True,USD
1,20261118T223104Z_AR,2026-11-18T22:31:04Z,federal,https://marketplace-int.api.healthcare.gov/api...,AR,71635,05003,2026,1,159600,...,POS,319.66,319.66,0.0,7500.0,None,True,10000.0,True,USD
2,20261118T223104Z_AR,2026-11-18T22:31:04Z,federal,https://marketplace-int.api.healthcare.gov/api...,AR,71635,05003,2026,1,159600,...,PPO,322.57,322.57,0.0,7500.0,None,True,10000.0,True,USD
3,20261118T223104Z_AR,2026-11-18T22:31:04Z,federal,https://marketplace-int.api.healthcare.gov/api...,AR,71635,05003,2026,1,159600,...,POS,330.93,330.93,0.0,7500.0,None,True,10000.0,True,USD
4,20261118T223104Z_AR,2026-11-18T22:31:04Z,federal,https://marketplace-int.api.healthcare.gov/api...,AR,71635,05003,2026,1,159600,...,POS,340.14,340.14,0.0,9975.0,None,True,9975.0,True,USD


## Compile

In [ ]:
import shutil

import dask.bag as db


def compile_run(raw_file: str) -> None:
    """One raw file -> CSV_DIR/<file name without .raw.jsonl>/<county_fips>.csv."""
    name = Path(raw_file).name.removesuffix(".jsonl").removesuffix(".raw")
    out_dir = CSV_DIR / name
    if out_dir.exists() and not OVERWRITE:
        print(f"{name}: already compiled, skipped")
        return
    lines = db.read_text(raw_file, blocksize=BLOCKSIZE)
    rows = pd.DataFrame(lines.map(json.loads).map(entry_to_rows).flatten().compute())  # parsed in parallel
    if rows.empty:
        print(f"{name}: no plans in the raw file, skipped")
        return
    shutil.rmtree(out_dir, ignore_errors=True)
    out_dir.mkdir(parents=True)
    for county_fips, county_rows in rows.groupby("county_fips"):
        county_rows.to_csv(out_dir / f"{county_fips}.csv", index=False)
    print(f"{name}: {len(rows):,} rows in {rows['county_fips'].nunique()} county files -> {out_dir}")


for raw_file in run_ids:
    compile_run(raw_file)

## Check

- Loads everything compiled as one dask dataframe
- Prints the partition count, then rows and counties per run and state

In [ ]:
import dask.dataframe as dd

if any(CSV_DIR.glob("*/*.csv")):
    compiled = dd.read_csv(str(CSV_DIR / "*" / "*.csv"), dtype={"zip_code": str, "county_fips": str, "plan_id": str}, assume_missing=True)
    print(compiled.npartitions, "partitions")
    per_county = compiled.groupby(["run_id", "state", "county_fips"]).size().compute()
    display(per_county.groupby(["run_id", "state"]).agg(rows="sum", counties="count").reset_index())
else:
    print(f"no compiled CSVs in {CSV_DIR} yet")